In [ ]:
import sys

!{sys.executable} -m pip install tavily-python groq

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime

from tavily import TavilyClient
from groq import Groq

In [ ]:
import getpass

TAVILY_API_KEY = getpass.getpass("Enter Tavily API Key: ")
GROQ_API_KEY = getpass.getpass("Enter Groq API Key: ")

In [ ]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

In [ ]:
def fetch_latest_gold_news(max_results=10):
    query = (
        "latest gold price news inflation interest rates dollar "
        "Federal Reserve safe haven demand geopolitical risk"
    )
    
    response = tavily_client.search(
        query=query,
        search_depth="advanced",
        max_results=max_results,
        include_answer=False,
        include_raw_content=False
    )
    
    results = response.get("results", [])
    
    rows = []
    for item in results:
        rows.append({
            "title": item.get("title"),
            "url": item.get("url"),
            "content": item.get("content"),
            "score": item.get("score")
        })
    
    return pd.DataFrame(rows)

In [ ]:
latest_news = fetch_latest_gold_news(max_results=10)

print(latest_news.shape)
latest_news.head(10)

In [ ]:
def format_news_for_llm(news_df):
    lines = []
    
    for i, row in news_df.iterrows():
        title = str(row.get("title", "")).strip()
        content = str(row.get("content", "")).strip()
        
        text = f"{i+1}. Title: {title}\nSummary: {content}"
        lines.append(text)
    
    return "\n\n".join(lines)

In [ ]:
news_text = format_news_for_llm(latest_news)

print(news_text[:1500])

In [ ]:
def analyze_gold_news_with_groq(news_text, groq_client):
    prompt = f"""
You are a financial news analyst for gold price forecasting.

Analyze the following gold-related news headlines and summaries.
Estimate their likely short-term impact on gold prices over the next 7 trading days.

IMPORTANT:
Return exactly ONE valid JSON object.
Do not return multiple alternatives.
Do not include markdown.
Do not include explanation outside JSON.
Do not include trailing text.

Use this exact JSON schema:

{{
  "overall_news_signal": "bullish/bearish/neutral/mixed",
  "gold_sentiment_score": 0.0,
  "inflation_risk_score": 0.0,
  "interest_rate_risk_score": 0.0,
  "dollar_pressure_score": 0.0,
  "safe_haven_demand_score": 0.0,
  "geopolitical_risk_score": 0.0,
  "market_uncertainty_score": 0.0,
  "short_term_gold_impact": "positive/negative/mixed/neutral",
  "confidence_score": 0.0,
  "key_drivers": ["driver 1", "driver 2", "driver 3"],
  "summary": "2-3 sentence summary"
}}

Scoring rules:
- gold_sentiment_score must be between -1 and 1.
- dollar_pressure_score must be between -1 and 1.
- all other numeric scores must be between 0 and 1.
- Positive gold_sentiment_score means bullish for gold.
- Negative gold_sentiment_score means bearish for gold.
- Negative dollar_pressure_score means dollar strength is pressuring gold.

News:
{news_text}
"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "You are a strict JSON generator. Return exactly one JSON object and nothing else."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=500
    )

    return response.choices[0].message.content.strip()

In [ ]:
groq_raw_output = analyze_gold_news_with_groq(
    news_text=news_text,
    groq_client=groq_client
)

print(groq_raw_output)

In [ ]:
def parse_groq_json(groq_output):
    try:
        return json.loads(groq_output)
    except json.JSONDecodeError:
        # Basic cleanup if model accidentally wraps JSON in text
        start = groq_output.find("{")
        end = groq_output.rfind("}") + 1
        
        if start != -1 and end != -1:
            cleaned = groq_output[start:end]
            return json.loads(cleaned)
        
        raise ValueError("Could not parse Groq output as JSON.")

In [ ]:
news_context = parse_groq_json(groq_raw_output)

news_context

In [ ]:
from datetime import datetime
import os
import json

news_context["analysis_timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
news_context["news_items_analyzed"] = len(latest_news)

os.makedirs("outputs", exist_ok=True)

with open("outputs/latest_gold_news_context.json", "w") as f:
    json.dump(news_context, f, indent=4)

print("Saved latest gold news context.")
news_context

In [ ]:
with open("outputs/latest_gold_prediction_summary.json", "r") as f:
    latest_gold_prediction_summary = json.load(f)

latest_gold_prediction_summary

In [ ]:
def combine_model_and_news_signal(prediction_summary, news_context):
    model_signal = prediction_summary["Risk Signal"]
    news_signal = news_context["overall_news_signal"]
    news_score = news_context["gold_sentiment_score"]
    
    if model_signal in ["Bullish / High Upside", "Moderately Bullish"] and news_score > 0.25:
        final_signal = "Bullish"
    elif model_signal in ["Bearish / Downside Risk", "Moderately Bearish"] and news_score < -0.25:
        final_signal = "Bearish"
    elif model_signal == "Neutral" and news_score < -0.25:
        final_signal = "Cautious Neutral / News Bearish"
    elif model_signal == "Neutral" and news_score > 0.25:
        final_signal = "Cautious Neutral / News Bullish"
    elif news_score < -0.5:
        final_signal = "Bearish News Risk"
    elif news_score > 0.5:
        final_signal = "Bullish News Support"
    else:
        final_signal = "Neutral / Mixed"
    
    combined_summary = {
        "Current Gold Price": prediction_summary["Current Gold Price"],
        "Predicted 7-Day Future Gold Price": prediction_summary["Predicted 7-Day Future Gold Price"],
        "Expected Change": prediction_summary["Expected Change"],
        "Expected Change (%)": prediction_summary["Expected Change (%)"],
        "Model Risk Signal": prediction_summary["Risk Signal"],
        "News Signal": news_signal,
        "News Sentiment Score": news_score,
        "Final Combined Signal": final_signal,
        "Key News Drivers": news_context["key_drivers"],
        "News Summary": news_context["summary"],
        "Final Recommendation": None
    }
    
    if final_signal == "Bullish":
        combined_summary["Final Recommendation"] = "Model and news both support upside; monitor confirmation before taking aggressive positions."
    elif final_signal == "Bearish":
        combined_summary["Final Recommendation"] = "Model and news both indicate downside pressure; monitor dollar strength, rates, and support levels."
    elif final_signal == "Cautious Neutral / News Bearish":
        combined_summary["Final Recommendation"] = "Model forecast is neutral, but news sentiment is bearish. Avoid aggressive bullish exposure and monitor inflation, Fed, and dollar signals."
    elif final_signal == "Cautious Neutral / News Bullish":
        combined_summary["Final Recommendation"] = "Model forecast is neutral, but news sentiment supports upside. Monitor safe-haven demand and dollar weakness."
    else:
        combined_summary["Final Recommendation"] = "Signals are mixed; continue monitoring price action and macro news."
    
    return combined_summary

In [ ]:
combined_gold_summary = combine_model_and_news_signal(
    latest_gold_prediction_summary,
    news_context
)

combined_gold_summary

In [ ]:
with open("outputs/combined_gold_forecast_summary.json", "w") as f:
    json.dump(combined_gold_summary, f, indent=4)

print("Saved combined gold forecast summary.")
print("Outputs folder:", os.listdir("outputs"))